In [2]:
import json

# Your input string
input_string = (
    "RELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKDRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKDRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKDRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKDRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKDRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKD"
)

# Initialize the dictionary
data = {}

# Iterate through the string and populate the dictionary
for index, char in enumerate(input_string, start=1):
    key = f"B{index}"
    data[key] = {char: 10.0}

# Write to a JSON file
with open("output.json", "w") as json_file:
    json.dump(data, json_file, indent=4)

print("JSON file 'output.json' has been created successfully.")

JSON file 'output.json' has been created successfully.


In [6]:
#!/usr/bin/env python3

import numpy as np

def parse_pdb(filepath):
    """
    Parses the PDB file and returns a list of lines,
    and a list of atom coordinates with their line indices.
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    atom_indices = []
    coordinates = []
    for i, line in enumerate(lines):
        if line.startswith(('ATOM  ', 'HETATM')):
            # Columns for x,y,z are 30-38, 38-46, 46-54 in PDB format
            try:
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                coordinates.append(np.array([x, y, z]))
                atom_indices.append(i)
            except ValueError:
                print(f"Warning: could not parse coordinates on line {i+1}")
    
    return lines, atom_indices, coordinates

def find_hetatom_coords(lines, chain_id, res_ids):
    """
    Finds the coordinates of HETATM for given chain and residue IDs.
    Returns a list of numpy arrays.
    """
    coords = []
    for line in lines:
        if line.startswith('HETATM'):
            # Columns: 22 for chain ID, 23-26 for residue sequence number
            line_chain = line[21].strip()
            try:
                res_id = int(line[22:26].strip())
            except ValueError:
                continue
            if line_chain == chain_id and res_id in res_ids:
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                coords.append(np.array([x, y, z]))
    return coords

def calculate_center_of_mass(coords):
    """
    Calculates the center of mass of given coordinates.
    Assuming equal mass for each point.
    """
    return np.mean(coords, axis=0)

def translate_coordinates(coordinates, translation_vector):
    """
    Applies translation to a list of numpy coordinate arrays.
    """
    return [coord + translation_vector for coord in coordinates]

def find_vector_between(coords1, coords2):
    """
    Returns the vector from coords1 to coords2.
    """
    return coords2 - coords1

def rotation_matrix_from_vectors(vec1, vec2):
    """
    Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return rot_matrix: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
    if np.allclose(a, b):
        return np.identity(3)
    if np.allclose(a, -b):
        # 180 degree rotation around any orthogonal vector
        # Find a vector orthogonal to a
        orthogonal = np.array([1, 0, 0])
        if np.allclose(a, orthogonal) or np.allclose(a, -orthogonal):
            orthogonal = np.array([0,1,0])
        v = np.cross(a, orthogonal)
        v /= np.linalg.norm(v)
        return rotation_matrix_axis_angle(v, np.pi)
    v = np.cross(a, b)
    s = np.linalg.norm(v)
    c = np.dot(a, b)
    if s == 0:
        return np.identity(3)
    vx = np.array([[0, -v[2], v[1]],
                  [v[2], 0, -v[0]],
                  [-v[1], v[0], 0]])
    rot_matrix = np.identity(3) + vx + np.matmul(vx, vx) * ((1 - c)/(s**2))
    return rot_matrix

def rotation_matrix_axis_angle(axis, angle):
    """
    Returns the rotation matrix associated with counterclockwise rotation about
    the given axis by the given angle (in radians) using Rodrigues' formula.
    """
    axis = axis / np.linalg.norm(axis)
    a = np.cos(angle / 2.0)
    b, c, d = -axis * np.sin(angle / 2.0)
    # Quaternion components
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    rot_matrix = np.array([[aa + bb - cc - dd, 2*(bc + ad), 2*(bd - ac)],
                           [2*(bc - ad), aa + cc - bb - dd, 2*(cd + ab)],
                           [2*(bd + ac), 2*(cd - ab), aa + dd - bb - cc]])
    return rot_matrix

def apply_rotation(coordinates, rot_matrix):
    """
    Applies rotation matrix to a list of numpy coordinate arrays.
    """
    return [np.dot(rot_matrix, coord) for coord in coordinates]

def update_pdb_lines(lines, atom_indices, new_coords):
    """
    Updates the PDB lines with new coordinates.
    """
    for idx, coord in zip(atom_indices, new_coords):
        # PDB format for coordinates: columns 30-38, 38-46, 46-54
        x, y, z = coord
        lines[idx] = lines[idx][:30] + f"{x:8.3f}{y:8.3f}{z:8.3f}" + lines[idx][54:]
    return lines

def main():
    pdb_path = '/home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes.pdb'
    chain_id = 'B'
    res_ids = [962, 1027]
    
    # Parse PDB
    lines, atom_indices, coordinates = parse_pdb(pdb_path)
    
    # Find hetatom coordinates
    hetatom_coords = find_hetatom_coords(lines, chain_id, res_ids)
    if len(hetatom_coords) != 2:
        print(f"Error: found {len(hetatom_coords)} hetatoms for chain {chain_id} residues {res_ids}")
        return
    
    coord1, coord2 = hetatom_coords
    # Calculate center of mass
    com = calculate_center_of_mass([coord1, coord2])
    
    # Translation vector to move COM to origin
    translation_vector = -com
    
    # Apply translation to all coordinates
    translated_coords = translate_coordinates(coordinates, translation_vector)
    
    # Update hetatom coordinates after translation
    translated_coord1 = coord1 + translation_vector
    translated_coord2 = coord2 + translation_vector
    
    # Find vector between hetatoms
    vector = find_vector_between(translated_coord1, translated_coord2)
    
    # Desired vector: align to z-axis
    desired_vector = np.array([0, 0, 1])
    
    # Compute rotation matrix
    rot_matrix = rotation_matrix_from_vectors(vector, desired_vector)
    
    # Apply rotation to all translated coordinates
    rotated_coords = apply_rotation(translated_coords, rot_matrix)
    
    # Update PDB lines with new coordinates
    updated_lines = update_pdb_lines(lines, atom_indices, rotated_coords)
    
    # Optionally, write out to a new PDB file
    output_path = pdb_path.replace('.pdb', '_transformed.pdb')
    with open(output_path, 'w') as f:
        f.writelines(updated_lines)
    
    print(f"Transformed PDB written to {output_path}")

if __name__ == "__main__":
    main()

Transformed PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_transformed.pdb


In [7]:
#!/usr/bin/env python3

import numpy as np

def parse_pdb(filepath):
    """
    Parses the PDB file and returns a list of lines,
    and a list of atom coordinates with their line indices.
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    atom_indices = []
    coordinates = []
    for i, line in enumerate(lines):
        if line.startswith(('ATOM  ', 'HETATM')):
            # Columns for x,y,z are 30-38, 38-46, 46-54 in PDB format
            try:
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                coordinates.append(np.array([x, y, z]))
                atom_indices.append(i)
            except ValueError:
                print(f"Warning: could not parse coordinates on line {i+1}")
    
    return lines, atom_indices, coordinates

def find_hetatom_coords(lines, chain_id, res_ids):
    """
    Finds the coordinates of HETATM for given chain and residue IDs.
    Returns a list of numpy arrays.
    """
    coords = []
    for line in lines:
        if line.startswith('HETATM'):
            # Columns: 22 for chain ID, 23-26 for residue sequence number
            line_chain = line[21].strip()
            try:
                res_id = int(line[22:26].strip())
            except ValueError:
                continue
            if line_chain == chain_id and res_id in res_ids:
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                coords.append(np.array([x, y, z]))
    return coords

def calculate_center_of_mass(coords):
    """
    Calculates the center of mass of given coordinates.
    Assuming equal mass for each point.
    """
    return np.mean(coords, axis=0)

def translate_coordinates(coordinates, translation_vector):
    """
    Applies translation to a list of numpy coordinate arrays.
    """
    return [coord + translation_vector for coord in coordinates]

def find_vector_between(coords1, coords2):
    """
    Returns the vector from coords1 to coords2.
    """
    return coords2 - coords1

def rotation_matrix_from_vectors(vec1, vec2):
    """
    Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return rot_matrix: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
    if np.allclose(a, b):
        return np.identity(3)
    if np.allclose(a, -b):
        # 180 degree rotation around any orthogonal vector
        # Find a vector orthogonal to a
        orthogonal = np.array([1, 0, 0])
        if np.allclose(a, orthogonal) or np.allclose(a, -orthogonal):
            orthogonal = np.array([0,1,0])
        v = np.cross(a, orthogonal)
        v /= np.linalg.norm(v)
        return rotation_matrix_axis_angle(v, np.pi)
    v = np.cross(a, b)
    s = np.linalg.norm(v)
    c = np.dot(a, b)
    if s == 0:
        return np.identity(3)
    vx = np.array([[0, -v[2], v[1]],
                  [v[2], 0, -v[0]],
                  [-v[1], v[0], 0]])
    rot_matrix = np.identity(3) + vx + np.matmul(vx, vx) * ((1 - c)/(s**2))
    return rot_matrix

def rotation_matrix_axis_angle(axis, angle):
    """
    Returns the rotation matrix associated with counterclockwise rotation about
    the given axis by the given angle (in radians) using Rodrigues' formula.
    """
    axis = axis / np.linalg.norm(axis)
    a = np.cos(angle / 2.0)
    b, c, d = -axis * np.sin(angle / 2.0)
    # Quaternion components
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    rot_matrix = np.array([[aa + bb - cc - dd, 2*(bc + ad), 2*(bd - ac)],
                           [2*(bc - ad), aa + cc - bb - dd, 2*(cd + ab)],
                           [2*(bd + ac), 2*(cd - ab), aa + dd - bb - cc]])
    return rot_matrix

def apply_rotation(coordinates, rot_matrix):
    """
    Applies rotation matrix to a list of numpy coordinate arrays.
    """
    return [np.dot(rot_matrix, coord) for coord in coordinates]

def update_pdb_lines(lines, atom_indices, new_coords):
    """
    Updates the PDB lines with new coordinates.
    """
    for idx, coord in zip(atom_indices, new_coords):
        # PDB format for coordinates: columns 30-38, 38-46, 46-54
        x, y, z = coord
        lines[idx] = lines[idx][:30] + f"{x:8.3f}{y:8.3f}{z:8.3f}" + lines[idx][54:]
    return lines

def main():
    pdb_path = '/home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes.pdb'
    chain_id = 'B'
    res_ids = [1170, 1234]
    
    # Parse PDB
    lines, atom_indices, coordinates = parse_pdb(pdb_path)
    
    # Find hetatom coordinates
    hetatom_coords = find_hetatom_coords(lines, chain_id, res_ids)
    if len(hetatom_coords) != 2:
        print(f"Error: found {len(hetatom_coords)} hetatoms for chain {chain_id} residues {res_ids}")
        return
    
    coord1, coord2 = hetatom_coords
    # Calculate center of mass
    com = calculate_center_of_mass([coord1, coord2])
    
    # Translation vector to move COM to origin
    translation_vector = -com
    
    # Apply translation to all coordinates
    translated_coords = translate_coordinates(coordinates, translation_vector)
    
    # Update hetatom coordinates after translation
    translated_coord1 = coord1 + translation_vector
    translated_coord2 = coord2 + translation_vector
    
    # Find vector between hetatoms
    vector = find_vector_between(translated_coord1, translated_coord2)
    
    # Desired vector: align to z-axis
    desired_vector = np.array([0, 0, 1])
    
    # Compute rotation matrix
    rot_matrix = rotation_matrix_from_vectors(vector, desired_vector)
    
    # Apply rotation to all translated coordinates
    rotated_coords = apply_rotation(translated_coords, rot_matrix)
    
    # Update PDB lines with new coordinates
    updated_lines = update_pdb_lines(lines, atom_indices, rotated_coords)
    
    # Optionally, write out to a new PDB file
    output_path = pdb_path.replace('.pdb', '_transformed2.pdb')
    with open(output_path, 'w') as f:
        f.writelines(updated_lines)
    
    print(f"Transformed PDB written to {output_path}")

if __name__ == "__main__":
    main()

Transformed PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_transformed2.pdb


In [11]:
#!/usr/bin/env python3

import numpy as np
import os

def parse_pdb(filepath):
    """
    Parses the PDB file and returns a list of lines,
    and a list of atom indices and their coordinates.
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    atom_indices = []
    coordinates = []
    for i, line in enumerate(lines):
        if line.startswith(('ATOM  ', 'HETATM')):
            try:
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                coordinates.append(np.array([x, y, z]))
                atom_indices.append(i)
            except ValueError:
                print(f"Warning: could not parse coordinates on line {i+1}")
    
    return lines, atom_indices, coordinates

def rotation_matrix_z(angle_degrees):
    """
    Creates a rotation matrix for rotation around the z-axis by the given angle in degrees.
    """
    angle_radians = np.deg2rad(angle_degrees)
    cos_a = np.cos(angle_radians)
    sin_a = np.sin(angle_radians)
    rot_matrix = np.array([
        [cos_a, -sin_a, 0],
        [sin_a,  cos_a, 0],
        [0,      0,     1]
    ])
    return rot_matrix

def apply_rotation(coordinates, rot_matrix):
    """
    Applies a rotation matrix to a list of numpy coordinate arrays.
    """
    return [rot_matrix @ coord for coord in coordinates]

def update_pdb_lines(lines, atom_indices, new_coords):
    """
    Updates the PDB lines with new coordinates and modifies chain IDs:
    - Chain A → Chain C
    - Chain B → Chain D
    """
    for idx, coord in zip(atom_indices, new_coords):
        x, y, z = coord
        line = lines[idx]

        # Modify chain ID (Column 22, index 21)
        original_chain_id = line[21]
        if original_chain_id == 'A':
            new_chain_id = 'C'
        elif original_chain_id == 'B':
            new_chain_id = 'D'
        else:
            new_chain_id = original_chain_id  # Leave unchanged for other chains

        # Replace chain ID in the line
        line = line[:21] + new_chain_id + line[22:]

        # Replace coordinates (Columns 31-38, 39-46, 47-54; indices 30:38, 38:46, 46:54)
        # Ensure that the new coordinates maintain the fixed-width formatting
        line = (
            line[:30] +
            f"{x:8.3f}{y:8.3f}{z:8.3f}" +
            line[54:]
        )

        # Update the line in the list
        lines[idx] = line

    return lines

def write_pdb(output_path, lines):
    """
    Writes the modified lines to a new PDB file.
    """
    with open(output_path, 'w') as f:
        f.writelines(lines)

def main():
    # Input PDB file
    input_pdb = '/home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_transformed2.pdb'
    
    if not os.path.isfile(input_pdb):
        print(f"Error: File {input_pdb} does not exist.")
        return
    
    # Parse PDB
    lines, atom_indices, coordinates = parse_pdb(input_pdb)
    
    # Define rotation angles
    angles = [-60, -90, -120]
    
    for angle in angles:
        rot_matrix = rotation_matrix_z(angle)
        rotated_coords = apply_rotation(coordinates, rot_matrix)
        updated_lines = update_pdb_lines(lines.copy(), atom_indices, rotated_coords)
        
        # Define output file name with rotation angle and chain modification
        base, ext = os.path.splitext(input_pdb)
        output_pdb = f"{base}_rotated_{angle}deg_modified{ext}"
        
        # Write rotated and modified PDB
        write_pdb(output_pdb, updated_lines)
        print(f"Rotated ({angle}° around z-axis) and chain-modified PDB written to {output_pdb}")

if __name__ == "__main__":
    main()

Rotated (-60° around z-axis) and chain-modified PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_transformed2_rotated_-60deg_modified.pdb
Rotated (-90° around z-axis) and chain-modified PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_transformed2_rotated_-90deg_modified.pdb
Rotated (-120° around z-axis) and chain-modified PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_transformed2_rotated_-120deg_modified.pdb


In [12]:
#!/usr/bin/env python3

import os

def read_pdb(filepath):
    """
    Reads a PDB file and returns its lines.
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    return lines

def filter_pdb_lines(lines):
    """
    Filters ATOM and HETATM lines from a list of PDB lines.
    """
    atom_lines = [line for line in lines if line.startswith(('ATOM  ', 'HETATM'))]
    return atom_lines

def combine_pdb_files(base_pdb, rotated_pdb, output_pdb):
    """
    Combines two PDB files by merging their atom lines.
    Ensures that only one END statement is present at the end.
    """
    base_lines = read_pdb(base_pdb)
    rotated_lines = read_pdb(rotated_pdb)

    # Filter atom lines
    base_atoms = filter_pdb_lines(base_lines)
    rotated_atoms = filter_pdb_lines(rotated_lines)

    # Combine atom lines
    combined_atoms = base_atoms + rotated_atoms

    # Prepare header from base_pdb (assuming it has header lines before ATOM/HETATM)
    # We'll take all lines up to the first ATOM/HETATM line as header
    header = []
    for line in base_lines:
        if line.startswith(('ATOM  ', 'HETATM')):
            break
        header.append(line)

    # Prepare footer (only one END statement)
    footer = ['END\n']

    # Combine all parts
    combined_pdb = header + combined_atoms + footer

    # Write to output PDB file
    with open(output_pdb, 'w') as f:
        f.writelines(combined_pdb)

    print(f"Combined PDB written to {output_pdb}")

def main():
    # Directory containing the PDB files
    pdb_dir = '/home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/'

    # Base PDB file
    base_pdb = os.path.join(pdb_dir, 'fd31_forshapes_transformed.pdb')

    if not os.path.isfile(base_pdb):
        print(f"Error: Base PDB file {base_pdb} does not exist.")
        return

    # List of rotated and modified PDB files to combine with base_pdb
    rotated_pdb_files = [
        'fd31_forshapes_transformed2_rotated_-60deg_modified.pdb',
        'fd31_forshapes_transformed2_rotated_-90deg_modified.pdb',
        'fd31_forshapes_transformed2_rotated_-120deg_modified.pdb'
    ]

    # Iterate over each rotated PDB file and combine with base_pdb
    for rotated_pdb in rotated_pdb_files:
        rotated_pdb_path = os.path.join(pdb_dir, rotated_pdb)
        
        if not os.path.isfile(rotated_pdb_path):
            print(f"Warning: Rotated PDB file {rotated_pdb_path} does not exist. Skipping.")
            continue

        # Define output PDB file name
        base_name = 'fd31_forshapes_combined'
        angle = rotated_pdb.split('_rotated_')[-1].split('deg')[0]
        output_pdb = os.path.join(pdb_dir, f"{base_name}_{angle}deg.pdb")

        # Combine the PDB files
        combine_pdb_files(base_pdb, rotated_pdb_path, output_pdb)

if __name__ == "__main__":
    main()

Combined PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_combined_-60deg.pdb
Combined PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_combined_-90deg.pdb
Combined PDB written to /home/pkwon2/2d_array/dhr31_ligand_mpnn/outputs/ligand/packed/fd31_forshapes_combined_-120deg.pdb
